## PydanticAI 기본 실습

**실습 목표:**

1. **PydanticAI의 Agent 개념** 을 이해하고 기본 텍스트 생성을 수행할 수 있다
2. **매개변수(temperature, max_tokens 등)** 를 조절하여 AI 응답을 제어할 수 있다
3. **스트리밍 응답** 으로 AI 답변을 실시간으로 출력할 수 있다
4. **멀티턴 대화** 를 구현하여 대화 맥락을 유지할 수 있다
5. **도구(Tool)** 를 등록하여 Agent의 기능을 확장할 수 있다
6. **비동기 동시 호출** 로 여러 요청을 병렬 처리할 수 있다

**사용 프레임워크:** [PydanticAI](https://ai.pydantic.dev/) — Python 에이전트 프레임워크

### AI Agent란?

**AI Agent** 는 단순히 질문에 답하는 챗봇을 넘어서, **스스로 판단하고 도구를 사용하여 작업을 수행하는 AI 시스템** 입니다.

![AI Agent는 지우와 피카츄의 관계와 같습니다](images/ai_agent_pokemon.png)

**포켓몬으로 이해하는 AI Agent**

| 포켓몬 세계 | AI Agent 세계 | 설명 |
|------------|-------------|------|
| **지우** (트레이너) | **개발자** (나) | 목표를 정하고 지시를 내리는 사람 |
| **피카츄** (포켓몬) | **Agent** (LLM + 도구 + 지침) | 지시를 받고 스스로 판단하여 행동하는 존재 |
| **기술** (10만볼트, 전광석화) | **도구 (Tool)** | Agent가 상황에 맞게 선택하여 호출하는 기능 |
| **지우의 전략/지시** | **시스템 프롬프트 (instruction)** | Agent가 어떻게 행동할지 정하는 지침 |

**일반 LLM 호출 vs AI Agent 비교:**

```
[일반 LLM 호출]  지우: "피카츄, 저기 뭐야?" => 피카츄: "피카" (대답만 함)

[AI Agent]       지우: "피카츄, 저 포켓몬 잡아!" 
                 => 피카츄가 스스로 판단: "날아다니니까 10만볼트!"  (도구 선택)
                 => 10만볼트 사용                                  (도구 실행)
                 => "잡았다! HP 35, 타입: 비행"                    (결과 반환)
```

| 구분 | 일반 LLM 호출 | AI Agent |
|------|-------------|----------|
| **동작 방식** | 질문 => 답변 (1회성) | 목표를 받고 => 판단 => 도구 사용 => 결과 반환 |
| **도구 사용** | 불가능 | 검색, 계산, API 호출 등 **외부 도구를 직접 호출** |
| **판단력** | 주어진 질문에만 답변 | 어떤 도구를 쓸지 **스스로 판단** |
| **상태 관리** | 이전 대화를 기억하지 못함 | 대화 맥락을 유지하며 **연속 작업 가능** |

> **이 실습에서는** PydanticAI 프레임워크를 사용하여 AI Agent를 만들고,
> 도구 등록, 구조화된 출력, 멀티턴 대화 등 Agent의 핵심 기능을 실습합니다.

### 환경 설정

실습에 필요한 패키지를 설치하고, API 키를 설정합니다.

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
# !pip install -q pydantic-ai-slim[google] python-dotenv pydantic

# uv 권장! uv 환경에서는 위 pip 대신 터미널에서 uv sync 명령어로 설치하세요:
# uv sync

**PydanticAI 모델별 설정 가이드**

PydanticAI는 다양한 LLM 프로바이더를 지원합니다. 각 프로바이더별 **모델 ID 형식** 과 **환경변수** 설정 방법은 아래와 같습니다.

| 프로바이더 | 모델 ID 형식 | 모델 예시 | 환경변수 (API Key) | 설치 패키지 |
|---|---|---|---|---|
| **Google Gemini** | `google-gla:{모델명}` | `google-gla:gemini-2.5-flash` | `GEMINI_API_KEY` | `pydantic-ai-slim[google]` |
| **OpenAI** | `openai:{모델명}` | `openai:gpt-4o`, `openai:gpt-4o-mini` | `OPENAI_API_KEY` | `pydantic-ai-slim[openai]` |
| **Anthropic (Claude)** | `anthropic:{모델명}` | `anthropic:claude-sonnet-4-20250514` | `ANTHROPIC_API_KEY` | `pydantic-ai-slim[anthropic]` |

**`.env` 파일 설정 예시:**

```
# Google Gemini
GEMINI_API_KEY=your-gemini-api-key

# OpenAI
OPENAI_API_KEY=your-openai-api-key

# Anthropic (Claude)
ANTHROPIC_API_KEY=your-anthropic-api-key
```

**코드 사용 예시:**

```python
from pydantic_ai import Agent

# Google Gemini
agent = Agent("google-gla:gemini-2.5-flash")

# OpenAI
agent = Agent("openai:gpt-4o")

# Anthropic Claude
agent = Agent("anthropic:claude-sonnet-4-20250514")
```

**공식 문서 링크:**

- [PydanticAI 모델 설정 개요](https://ai.pydantic.dev/models/)
- [PydanticAI - Google Gemini 설정](https://ai.pydantic.dev/models/google/)
- [PydanticAI - OpenAI 설정](https://ai.pydantic.dev/models/openai/)
- [PydanticAI - Anthropic (Claude) 설정](https://ai.pydantic.dev/models/anthropic/)

**실습**
- 환경변수를 로드하고 API 키가 올바르게 설정되었는지 확인합니다.
- `.env` 파일에 `GEMINI_API_KEY`를 설정한 후 아래 셀을 실행합니다.

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
from pprint import pprint
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

### PydanticAI란?

- [PydanticAI 공식 문서](https://ai.pydantic.dev/)

PydanticAI는 **생성형 AI 애플리케이션을 쉽게 만들기 위한 Python 에이전트 프레임워크** 입니다.

웹 개발에서 FastAPI가 API 서버를 쉽게 만들어주는 것처럼, PydanticAI는 AI 에이전트를 쉽게 만들어줍니다.

**핵심 개념:**

| 개념 | 설명 | 비유 |
|------|------|------|
| **Agent** | AI와 대화하는 단위. 모델, 시스템 프롬프트, 도구를 하나로 묶음 | 특정 역할을 하는 직원 |
| **Structured Output** | AI 응답을 Pydantic 모델(스키마)로 강제 | 정해진 양식에 맞춰 보고서 작성 |
| **Tool** | Agent가 호출할 수 있는 Python 함수 | 직원이 사용하는 도구(계산기, DB 등) |
| **Dependency Injection** | 실행 시점에 필요한 데이터를 주입 | 직원에게 업무에 필요한 자료 전달 |

### 1. 기본 텍스트 생성 (Single-turn)

- [Agent 공식 문서](https://ai.pydantic.dev/agents/)

**(1) Agent와 실행 방식**

PydanticAI에서 AI와 대화하려면 먼저 **Agent** 를 만들어야 합니다.

```python
agent = Agent('google-gla:gemini-3.1-flash-lite-preview')  # Agent 생성 (모델 지정)
result = await agent.run('질문 내용')                        # 비동기 실행
print(result.output)                                        # 응답 텍스트
```

**실행 방식 3가지:**

| 메서드 | 방식 | 사용 상황 |
|--------|------|----------|
| `agent.run_sync()` | 동기(blocking) | 일반 Python 스크립트 (이벤트 루프 없을 때) |
| `await agent.run()` | 비동기(async) | 동시 호출 시, Jupyter 노트북 |
| `await agent.run_stream()` | 비동기 스트리밍 | 실시간 응답 출력 |

> **참고:** Jupyter 노트북은 내부적으로 이벤트 루프가 이미 실행 중이므로 `run_sync()`를 사용하면 충돌이 발생합니다. Jupyter에서는 반드시 `await agent.run()`을 사용합니다.

**실습 (1)**
- Agent를 생성하고 기본 질문을 보내봅니다.
- `Agent(model_id)`로 Agent를 만들고, `await agent.run()`으로 질문합니다.

In [ ]:
from pydantic_ai import Agent

# Agent 생성 - 모델만 지정하면 바로 사용 가능
agent = Agent(model_id) # 모델 ID는 어디서 확인??? gemini 공식문서에 모델 ID

# 기본 텍스트 생성 요청 (비동기 방식 — Jupyter에서는 await 사용)
search_query = "스파르타 부트캠프 데이터 분석 트랙에 대해서 알려줘"

result = await agent.run(search_query) # 비동기 함수의 작업이 완료까지 기다린다!

# 응답 출력
print(f"질문: {search_query}")
print(f"답변: {result.output}")
print("-" * 50)

**실습 (2)**
- 응답 객체의 내부 구조를 확인합니다.
- `result.output`, `result.usage()`, `result.new_messages()`를 살펴봅니다.

In [ ]:
result
type(result) #pydantic_ai.run.AgentRunResult

In [ ]:
# 응답 객체를 더 자세히 살펴봅시다
print("=" * 50)
print("[응답 객체 구조]")
print("=" * 50)

# 1. 응답 텍스트
print(f"\n[output] 응답 텍스트:")
print(result.output[:200], "...")

# 2. 토큰 사용량
print(f"\n[usage] 토큰 사용량:")
pprint(result.usage())
# RunUsage(input_tokens=17, output_tokens=897, details={'text_prompt_tokens': 17}, requests=1)
# 비용 계산!

# 3. 대화 메시지 기록 (멀티턴에서 활용)
print(f"\n[new_messages] 대화 기록: {len(result.new_messages())}개")
for msg in result.new_messages():
    print(f"  - {type(msg).__name__}: {str(msg)[:100]}...")

**(2) 토큰 사용량 확인 및 비용 산출**

- [Gemini 모델 비용 문서 링크](https://ai.google.dev/gemini-api/docs/pricing?hl=ko)

PydanticAI에서는 `result.usage()` 메서드로 토큰 사용량을 확인합니다.

```python
usage = result.usage()
usage.input_tokens    # 입력 토큰 수
usage.output_tokens   # 출력 토큰 수
```

**실습 (3)**
- 위 실행 결과의 토큰 사용량과 비용을 계산합니다.
- Gemini 모델의 토큰당 가격을 기준으로 USD/KRW 비용을 산출합니다.

In [ ]:
# 토큰 사용량 정보
usage = result.usage()
print(usage)

prompt_tokens = usage.input_tokens or 0     # 입력 토큰 수
output_tokens = usage.output_tokens or 0    # 응답 토큰 수

# gemini-3.1-flash-lite 가격 (2026-03-23 기준, 백만 토큰당 USD)
input_price = 0.25 #백만 토큰 당 가격
output_price = 0.50 #백만 토큰 당 가격
exchange_rate = 1500  # 환율

# 비용 계산 (USD)
input_cost_usd = (prompt_tokens / 1_000_000) * input_price
output_cost_usd = (output_tokens / 1_000_000) * output_price
total_cost_usd = input_cost_usd + output_cost_usd

# 비용 계산 (KRW)
total_cost_krw = total_cost_usd * exchange_rate

print(f"\n" + "=" * 60)
print(f"{'토큰 사용량 및 비용':^52}")
print("=" * 60)
print(f"프롬프트 토큰: {prompt_tokens:>6,} tokens")
print(f"              ${input_cost_usd:>9.6f}  (₩{input_cost_usd * exchange_rate:>7.4f})")
print(f"-" * 60)
print(f"응답 토큰:     {output_tokens:>6,} tokens")
print(f"              ${output_cost_usd:>9.6f}  (₩{output_cost_usd * exchange_rate:>7.4f})")
print(f"-" * 60)
print(f"총 비용:      ${total_cost_usd:>9.6f}  (₩{total_cost_krw:>7.4f})")
print("=" * 60)

# 최종 프로젝트에서 LLM API사용시 비용계산!
# 내 프로젝트에서 얼마나 비용이들지 !
# - 기업에서도 분석계획 <-- 비용도 넣어야 승인!

# 데이터 1000건을 LLM을 써야한다

# 약 10건에 대해서 실행해서 -> 1회 호출당 입력,출력 평균 토큰을 얻고
# 1회 호출시 비용 * 1000 ==> 1000회에 따른 대략적인 비용산출이 !!!

**(3) 매개변수를 활용한 상세 설정**

- [Pydantic AI - Google 모델 설정 문서](https://ai.pydantic.dev/models/google/)
- [Gemini API 가격 정책](https://ai.google.dev/gemini-api/docs/pricing?hl=ko)

PydanticAI에서는 `model_settings` 매개변수로 AI의 동작을 제어합니다.

![LLM 단어 생성 과정](images/LLM_단어생성.gif)

| 매개변수 | 설명 | 기본값 | 범위 |
|----------|------|--------|------|
| `temperature` | 창의성 조절 (높을수록 다양한 응답) | 1.0 | 0.0 ~ 2.0 |
| `max_tokens` | 최대 출력 토큰 수 | 모델별 상이 | 1 ~ 모델 한도 |
| `top_p` | 다양성 조절 (누적 확률 기반) | 0.95 | 0.0 ~ 1.0 |

**설정 방법 2가지:**

```python
# 방법 1: dict로 설정
result = await agent.run(query, model_settings={'temperature': 0.5, 'max_tokens': 200})

# 방법 2: GoogleModelSettings 객체 (Google 전용 설정 포함)
from pydantic_ai.models.google import GoogleModelSettings
settings = GoogleModelSettings(temperature=0.5, max_tokens=200)
result = await agent.run(query, model_settings=settings)
```

**Thinking 설정 (`google_thinking_config`)**

Gemini 모델의 **추론(thinking) 깊이** 를 조절할 수 있습니다. 모델 세대에 따라 설정 방식이 다릅니다.

| 모델 세대 | 설정 키 | 값 | 설명 |
|-----------|---------|-----|------|
| **Gemini 3** 이상 | `thinking_level` | `'low'`, `'high'` | 추론 깊이 수준 선택 |
| **Gemini 2.5** | `thinking_budget` | `0` ~ 정수값 | 추론 토큰 예산 (0 = 비활성화) |

```python
# Gemini 3: thinking_level로 추론 깊이 조절
settings = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'low'}
)

# Gemini 2.5: thinking_budget으로 추론 토큰 수 제한
settings = GoogleModelSettings(
    google_thinking_config={'thinking_budget': 0}  # 0이면 thinking 비활성화
)
```

**실습 (4)**
- 시스템 프롬프트와 매개변수를 설정하여 AI 응답을 제어합니다.
- `GoogleModelSettings`로 temperature, max_tokens, top_p를 조절합니다.
- [GoogleModelSettings 공식 문서](https://ai.pydantic.dev/models/google/) | [Gemini API 생성 파라미터 가이드](https://ai.google.dev/gemini-api/docs/text-generation?hl=ko#configure)

In [ ]:
from pydantic_ai.models.google import GoogleModelSettings

# 시스템 프롬프트가 포함된 Agent 생성
expert_agent = Agent(
    model_id,
    system_prompt="너는 AI 기술을 쉽게 설명하는 전문가야. 초보자도 이해할 수 있게 설명해줘."
)

# 매개변수 상세 설정
settings = GoogleModelSettings(
    temperature=0.5,     # 낮은 창의성 => 일관된 응답
    max_tokens=200,      # 최대 200 토큰
    top_p=0.9,           # 다양성 조절
)

# 질문 설정
question = """
LLM(Large Language Model)과 VLM(Vision Language Model)의 차이점을 설명해주세요.

조건:
- 각 모델이 무엇인지 한 문장으로 정의
- 주요 차이점 3가지를 명확히 구분
- 각각의 실제 활용 예시 1개씩 포함
- 전체 5문장 이내로 작성
"""

try:
    result = await expert_agent.run(question, model_settings=settings)

    print(f"질문: {question}")
    print(f"모델: {model_id}")
    print(f"설정: 창의성={settings.temperature}, 다양성={settings.top_p}")
    print("=" * 80)
    print(f"\n응답:")
    print(result.output)

except Exception as e:
    print(f"API 호출 오류: {e}")

**실습 (5)**
- `thinking_level` 설정에 따른 추론 깊이 차이를 비교합니다.
- 같은 수학 문제를 `low`와 `high`로 풀어보고, 소요 시간과 토큰 사용량을 비교합니다.

In [ ]:
# Thinking 설정 비교: thinking_level에 따른 응답 차이 확인
import time

thinking_agent = Agent(
    model_id,
    system_prompt="수학 문제를 단계별로 풀어주는 전문가입니다."
)

math_question = "한 농부가 닭과 토끼를 합쳐 20마리를 키우는데, 다리 수의 합이 56개입니다. 닭과 토끼는 각각 몇 마리인가요?"

# thinking_level='low' => 빠르지만 간단한 추론
settings_low = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'low'},
    temperature=0.3,
)

# thinking_level='high' => 느리지만 깊은 추론
settings_high = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'high'},
    temperature=0.3,
)

In [ ]:
# low 모드 실행
start = time.time()
result_low = await thinking_agent.run(math_question, model_settings=settings_low)
time_low = time.time() - start

print(f"질문: {math_question}")
print("=" * 80)
print(f"[thinking_level='low'] (소요시간: {time_low:.2f}초)")
print("-" * 40)
print(result_low.output)

In [ ]:
# high 모드 실행
start = time.time()
result_high = await thinking_agent.run(math_question, model_settings=settings_high)
time_high = time.time() - start

print(f"[thinking_level='high'] (소요시간: {time_high:.2f}초)")
print("-" * 40)
print(result_high.output)

In [ ]:
# 토큰 사용량 비교
usage_low = result_low.usage()
usage_high = result_high.usage()

print("=" * 50)
print("토큰 사용량 비교")
print("=" * 50)
print(f"{'':>15} {'low':>15} {'high':>15}")
print(f"{'입력 토큰':>15} {usage_low.input_tokens or 0:>15,} {usage_high.input_tokens or 0:>15,}")
print(f"{'출력 토큰':>15} {usage_low.output_tokens or 0:>15,} {usage_high.output_tokens or 0:>15,}")

### 2. 스트리밍 응답

**일반 응답 방식 (기본)**
- AI가 답변을 **완전히 다 만든 후** 에 한 번에 보여줍니다
- 긴 답변일수록 기다리는 시간이 길어집니다

**스트리밍 응답 방식 (개선)**
- AI가 답변을 **만드는 동시에 조금씩** 보여줍니다
- 사용자가 기다림 없이 답변을 바로 읽을 수 있어 더 자연스럽습니다

PydanticAI에서는 `run_stream()` + `stream_text()` 를 사용합니다.

```python
async with agent.run_stream(question) as result:
    async for text in result.stream_text(delta=True):
        print(text, end="")
```

> **참고:** `run_stream()`은 비동기(async) 전용입니다. Jupyter 노트북에서는 `await`로 바로 실행할 수 있습니다.

**실습**
- `run_stream()`으로 AI 응답을 실시간으로 출력합니다.
- `delta=True` 옵션으로 새로 생성된 텍스트만 스트리밍합니다.

In [ ]:
question = "AI가 어떻게 작동하는지 간단히 설명해주세요"
print(f"질문: {question}")
print("\n실시간 응답:")

# delta=True: 이전 텍스트를 포함하지 않고 새로 생성된 부분만 출력
async with agent.run_stream(question) as result:
    async for text in result.stream_text(delta=True):
        print(text, end="", flush=True)

print(f"\n\n사용 토큰: {result.usage()}")

### 3. 멀티턴 대화 (Multi-turn Conversation)

**멀티턴 대화란?**
- **Single-turn**: 한 번의 질문과 한 번의 응답으로 완결. 이전 대화 맥락을 기억하지 않음
- **Multi-turn**: 여러 차례 주고받으며 **이전 대화 맥락을 유지** 하는 대화 방식
- 예: "데이터 분석 취업 준비 방법" => "그중 먼저 할 것은?" => AI가 앞 대화를 기억하고 답변

PydanticAI에서 멀티턴 대화는 `message_history` 매개변수로 구현합니다.

**원리:**
1. 첫 번째 질문의 결과에서 `result.new_messages()` 로 대화 기록을 가져옵니다
2. 다음 질문에 `message_history=이전_메시지` 를 전달합니다
3. AI가 이전 대화 맥락을 기억한 상태로 응답합니다

```python
# 리스트에 누적 (대화가 길어질 때 편리)
history = []
result1 = await agent.run('첫 번째 질문', message_history=history)
history.extend(result1.new_messages())
result2 = await agent.run('두 번째 질문', message_history=history)
history.extend(result2.new_messages())
```

> **참고:** `new_messages()`는 해당 턴에서 새로 생긴 메시지만 반환합니다. 대화 기록을 직접 관리하므로 중간 수정, 분기, 저장/복원이 자유롭습니다.

**실습 (1)**
- 3턴 대화를 통해 `message_history`로 맥락을 유지하는 방법을 익힙니다.
- 이전 대화 기록을 합쳐서 전달하면 AI가 앞 대화를 기억합니다.

In [ ]:
# 시스템 프롬프트가 포함된 전문가 Agent 생성
chat_agent = Agent(
    model_id,
    system_prompt="너는 데이터 분석 분야의 시니어 전문가야. 실무 경험을 바탕으로 구체적이고 실용적인 조언을 제공해."
)

settings = GoogleModelSettings(temperature=0.8, max_tokens=500)

# 첫 번째 메시지
print("[첫 번째 대화]")
user_msg1 = "저는 지금 데이터 분석 직무에 취업을 하려해요."
print(f"사용자: {user_msg1}")

result1 = await chat_agent.run(user_msg1, model_settings=settings)
print(f"AI 전문가: {result1.output}")

# 두 번째 메시지 — message_history로 이전 대화 맥락 전달
print(f"\n[두 번째 대화]")
user_msg2 = "제가 가장 먼저 무엇을 준비하면 좋을까요?"
print(f"사용자: {user_msg2}")

result2 = await chat_agent.run(
    user_msg2,
    message_history=result1.new_messages(),  # 이전 대화 기록 전달!
    model_settings=settings
)
print(f"AI 전문가: {result2.output}")

# 세 번째 메시지 — 누적된 대화 기록 전달
print(f"\n[세 번째 대화]")
user_msg3 = "Python은 어느 정도 수준까지 알아야 하나요?"
print(f"사용자: {user_msg3}")

# 이전 두 대화의 기록을 합쳐서 전달
all_messages = result1.new_messages() + result2.new_messages()
result3 = await chat_agent.run(
    user_msg3,
    message_history=all_messages,
    model_settings=settings
)
print(f"AI 전문가: {result3.output}")

**실습 (2)**
- 스트리밍 + 멀티턴을 결합한 대화형 채팅 루프를 실행합니다.
- `input()`으로 사용자 입력을 받고, 스트리밍으로 실시간 응답합니다.

**주의:** 이 셀은 `input()`을 사용하므로 **Run All로 전체 실행하면 여기서 멈춥니다.** 반드시 이 셀만 개별 실행하세요.

In [ ]:
# 실시간 스트리밍 + 멀티턴 대화 루프
# '종료' 또는 'quit' 입력 시 대화 종료

stream_chat_agent = None

print("실시간 PydanticAI 채팅 (스트리밍)")
print("=" * 60)
print("명령어: '종료' 또는 'quit' => 채팅 종료")
print("=" * 60)

message_history = []  # 대화 기록 누적
turn_count = 0

while True:
    # TODO : 스트리밍 + 멀티턴 대화 구현
    pass

### 4. 도구 (Tool)

- [도구(Tool) 공식 문서](https://ai.pydantic.dev/tools/)

Agent에게 **Python 함수를 도구로 등록** 하면, AI가 필요할 때 직접 함수를 호출하여 정보를 가져옵니다.

**동작 흐름:**

![도구(Tool) 동작 흐름](images/tool_flow.png)

1. 사용자가 질문을 보냅니다
2. AI가 질문을 분석하여 **어떤 도구가 필요한지 자동으로 판단** 합니다
3. 선택된 도구(Python 함수)를 **AI가 직접 호출** 합니다
4. 도구 함수의 반환값을 받아 **최종 응답을 생성** 합니다

**실습 (1)**
- 날짜 조회, 외부 API(Wikipedia, YouTube)를 도구로 연결하여 실시간 정보를 검색합니다.
- **사전 준비:** `.env` 파일에 `YOUTUBE_API_KEY`를 설정해야 YouTube 검색이 동작합니다.

In [ ]:
import wikipediaapi
from googleapiclient.discovery import build
from datetime import date 

# 위키피디아 클라이언트 초기화 (한국어)
wiki = wikipediaapi.Wikipedia(
    user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    language='ko'
)

# YouTube Data API 클라이언트 초기화 (.env의 YOUTUBE_API_KEY 사용)
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))

# 외부 API를 활용하는 도구가 포함된 Agent
search_agent = Agent(
    model_id,
    system_prompt=(
        "당신은 정보 검색 어시스턴트입니다. "
        "위키피디아와 YouTube 도구를 활용하여 정확한 정보를 제공하세요."
    ),
)

@search_agent.tool_plain
def get_current_date() -> str:
    """오늘 날짜를 반환합니다."""
    return str(date.today())

@search_agent.tool_plain
def search_wikipedia(query: str) -> str:
    """위키피디아에서 검색어에 해당하는 문서의 요약을 반환합니다."""
    page = wiki.page(query)
    if page.exists():
        summary = page.summary[:500]
        url = page.fullurl
        return f"[{page.title}]\n{summary}\n\n참조: {url}"
    return f"'{query}'에 대한 위키피디아 문서를 찾을 수 없습니다."


@search_agent.tool_plain
def search_youtube(query: str, max_results: int = 3) -> str:
    """YouTube에서 영상을 검색하여 제목과 링크를 반환합니다."""
    try:
        response = youtube.search().list(
            q=query,
            part='snippet',
            type='video',
            maxResults=max_results
        ).execute()

        results = []
        for item in response.get('items', []):
            title = item['snippet']['title']
            video_id = item['id']['videoId']
            url = f"https://www.youtube.com/watch?v={video_id}"
            results.append(f"- {title}\n  {url}")
        return "\n".join(results) if results else "검색 결과가 없습니다."
    except Exception as e:
        return f"YouTube 검색 오류: {e}"


# 외부 API 도구를 활용하는 질문들
questions = [
    "오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.",
    "YouTube에서 '파이썬 데이터 분석 입문' 영상을 찾아주세요.",
]

for q in questions:
    result = await search_agent.run(q)
    print(f"Q: {q}")
    print(f"A: {result.output}")
    print("-" * 50)

**실습 (2)**
- 어떤 도구가 호출되었는지 `result.new_messages()`로 확인합니다.
- AI가 도구를 호출한 전체 과정(요청 => 도구 호출 => 응답)을 살펴봅니다.

In [ ]:
# 어떤 툴이 호출되었는지 확인
result.new_messages()

### 5. 비동기 동시 호출 (Concurrent Requests)

**(1) 동기(Synchronous) vs 비동기(Asynchronous) 개념**

| | 동기 (Synchronous) | 비동기 (Asynchronous) |
|---|---|---|
| **동작 방식** | 작업이 끝날 때까지 기다렸다가 다음 작업 실행 | 작업을 보내놓고 다른 작업을 먼저 진행 |
| **장점** | 코드가 단순하고 직관적 | 여러 작업을 동시에 처리하여 시간 절약 |
| **단점** | 여러 작업을 순차적으로만 처리 (느림) | 코드가 조금 더 복잡 (`async`/`await` 필요) |

**Python 비동기 핵심 문법:**

| 문법 | 역할 |
|------|------|
| `async def` | 비동기 함수 정의 |
| `await` | 비동기 작업이 완료될 때까지 대기 |
| `asyncio.gather()` | 여러 비동기 작업을 동시에 실행 |

**(2) `asyncio.gather`를 활용한 동시 호출**

```python
# 동기 방식: 3개 요청 => 순차 실행 => 총 9초
result1 = agent.run_sync('질문1')  # 3초
result2 = agent.run_sync('질문2')  # 3초
result3 = agent.run_sync('질문3')  # 3초

# 비동기 동시 호출: 3개 요청 => 동시 실행 => 총 3초
result1, result2, result3 = await asyncio.gather(
    agent.run('질문1'),
    agent.run('질문2'),
    agent.run('질문3'),
)
```

> **참고:** `run_sync()`는 동시 호출이 불가능합니다. 비동기 `run()`만 `asyncio.gather`와 함께 사용할 수 있습니다.

**실습 (1)**
- `asyncio.gather()`로 3개 질문을 동시에 호출하는 기본 예시입니다.
- 한 줄로 여러 요청을 동시에 보내고, 결과를 리스트로 받습니다.

In [ ]:
import asyncio
import time

concurrent_agent = Agent(
    model_id,
    system_prompt="질문에 3문장 이내로 간결하게 답하세요."
)
settings = GoogleModelSettings(temperature=0.7, max_tokens=200)

start = time.time()
concurrent_results = await asyncio.gather(
    concurrent_agent.run("1+1?", model_settings=settings), 
     concurrent_agent.run("2+2?", model_settings=settings),
     concurrent_agent.run("3+3?", model_settings=settings)
)

pprint(concurrent_results)

**실습 (2)**
- 순차 실행과 동시 실행의 속도 차이를 직접 측정하여 비교합니다.
- 5개 질문을 하나씩 처리하는 것과 `asyncio.gather()`로 한꺼번에 처리하는 것의 시간 차이를 확인합니다.

In [ ]:
# 비교를 위한 Agent 생성
concurrent_agent = Agent(
    model_id,
    system_prompt="질문에 3문장 이내로 간결하게 답하세요."
)

questions = [
    "Python의 장점은?",
    "데이터 분석에서 SQL이 중요한 이유는?",
    "머신러닝과 딥러닝의 차이점은?",
    "판다스(Pandas) 라이브러리란?",
    "A/B 테스트란 무엇인가?",
]

settings = GoogleModelSettings(temperature=0.7, max_tokens=200)

In [ ]:
# 순차 실행 (하나씩 기다리며 실행)
print("[순차 실행] 5개 요청을 하나씩 처리")
print("=" * 60)

start = time.time()
sequential_results = []
for q in questions:
    r = await concurrent_agent.run(q, model_settings=settings)
    sequential_results.append(r)
seq_time = time.time() - start

for q, r in zip(questions, sequential_results):
    print(f"Q: {q}")
    print(f"A: {r.output[:80]}...")
    print()

print(f"순차 실행 소요 시간: {seq_time:.2f}초")

In [ ]:
# 동시 실행 (asyncio.gather로 병렬 처리)
print("[동시 실행] 5개 요청을 한꺼번에 처리")
print("=" * 60)

start = time.time()
concurrent_results = await asyncio.gather(
    *[concurrent_agent.run(q, model_settings=settings) for q in questions]
)
con_time = time.time() - start

for q, r in zip(questions, concurrent_results):
    print(f"Q: {q}")
    print(f"A: {r.output[:80]}...")
    print()

print(f"동시 실행 소요 시간: {con_time:.2f}초")

In [ ]:
# 결과 비교
print("=" * 50)
print("[결과 비교]")
print("=" * 50)
print(f"  순차 실행: {seq_time:.2f}초")
print(f"  동시 실행: {con_time:.2f}초")
if seq_time > con_time:
    print(f"  시간 절약: {seq_time - con_time:.2f}초 ({(1 - con_time/seq_time)*100:.0f}% 단축)")